# Member 2 — Trip Duration Prediction

## Step 1: task setup, split verification, and leakage review

This step verifies the existing chronological split artifacts and defines the duration-prediction feature boundaries before any preprocessing or model fitting. The large Parquet files are inspected through metadata, and only small train/validation samples are loaded. The test split is used only for file/schema metadata and pickup-time boundary verification; its target values are not read or used for modelling decisions.

**Target:** `trip_duration_minutes`

No split membership, source data, preprocessing, estimator, or model is changed in this step.

### 1. Split files, schemas, and exact row counts

Parquet metadata provides exact row counts and schemas without loading the full datasets into pandas. The file snapshots also provide a read-only safety check.

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / 'data' / 'splits').exists():
    ROOT = ROOT.parent

split_paths = {
    'train': ROOT / 'data' / 'splits' / 'train.parquet',
    'validation': ROOT / 'data' / 'splits' / 'validation.parquet',
    'test': ROOT / 'data' / 'splits' / 'test.parquet',
}
expected_rows = {'train': 31_988_176, 'validation': 6_814_901, 'test': 6_730_257}
expected_total = 45_533_334
target = 'trip_duration_minutes'
pickup_time_col = 'pickup_timestamp'

source_snapshots_before = {
    name: (path.stat().st_size, path.stat().st_mtime_ns)
    for name, path in split_paths.items() if path.exists()
}
split_info = {}
for name, path in split_paths.items():
    exists = path.is_file()
    if not exists:
        split_info[name] = {'exists': False, 'rows': None, 'target': False}
        continue
    parquet_file = pq.ParquetFile(path)
    names = parquet_file.schema_arrow.names
    split_info[name] = {
        'exists': True,
        'rows': parquet_file.metadata.num_rows,
        'row_groups': parquet_file.num_row_groups,
        'target': target in names,
        'pickup_timestamp': pickup_time_col in names,
        'columns': names,
    }
    print(f"{name:10s} | exists={exists} | rows={split_info[name]['rows']:,} | "
          f"row_groups={split_info[name]['row_groups']} | target={split_info[name]['target']}")

row_counts_match = all(
    split_info[name]['exists'] and split_info[name]['rows'] == expected_rows[name]
    for name in expected_rows
)
actual_total = sum(split_info[name]['rows'] or 0 for name in expected_rows)
split_verification_pass = row_counts_match and actual_total == expected_total
target_available_pass = all(split_info[name].get('target', False) for name in split_paths)
print(f'Exact total rows: {actual_total:,} (expected {expected_total:,})')
print('Split verification:', 'PASS' if split_verification_pass else 'FAIL')
print('Target availability:', 'PASS' if target_available_pass else 'FAIL')
assert split_verification_pass, 'Split files or exact metadata row counts do not match documented values.'
assert target_available_pass, f'{target} is missing from one or more split schemas.' 

train      | exists=True | rows=31,988,176 | row_groups=178 | target=True
validation | exists=True | rows=6,814,901 | row_groups=39 | target=True
test       | exists=True | rows=6,730,257 | row_groups=36 | target=True
Exact total rows: 45,533,334 (expected 45,533,334)
Split verification: PASS
Target availability: PASS


### 2. Chronological boundary verification

Pickup-time minimum and maximum statistics are read from Parquet row-group metadata. This confirms non-overlapping chronological ordering without reading the test target or loading full timestamp columns into pandas.

In [2]:
def parquet_timestamp_bounds(path: Path, column_name: str):
    parquet_file = pq.ParquetFile(path)
    column_index = parquet_file.schema_arrow.names.index(column_name)
    minima, maxima = [], []
    for row_group_index in range(parquet_file.num_row_groups):
        stats = parquet_file.metadata.row_group(row_group_index).column(column_index).statistics
        if stats is None or not stats.has_min_max:
            raise RuntimeError(f'Missing min/max metadata for {path.name}, row group {row_group_index}.')
        minima.append(pd.Timestamp(stats.min))
        maxima.append(pd.Timestamp(stats.max))
    return min(minima), max(maxima)

chronology = {name: parquet_timestamp_bounds(path, pickup_time_col) for name, path in split_paths.items()}
for name, (minimum, maximum) in chronology.items():
    print(f'{name:10s} | pickup min={minimum} | pickup max={maximum}')

chronology_pass = (
    chronology['train'][1] < chronology['validation'][0]
    and chronology['validation'][1] < chronology['test'][0]
)
print('Chronological order train < validation < test:', 'PASS' if chronology_pass else 'FAIL')
print('Test access: schema/row-count metadata and pickup_timestamp min/max metadata only; no test target values read.')
assert chronology_pass, 'The documented chronological splits overlap or are out of order.' 

train      | pickup min=2025-04-01 00:00:00 | pickup max=2025-12-10 23:59:59
validation | pickup min=2025-12-11 00:00:00 | pickup max=2026-02-04 23:59:58
test       | pickup min=2026-02-05 00:00:01 | pickup max=2026-03-31 23:59:59
Chronological order train < validation < test: PASS
Test access: schema/row-count metadata and pickup_timestamp min/max metadata only; no test target values read.


### 3. Member 1 contract and handover review

The feature contract governs what can be known before a trip starts. The lists below are candidate categories for later modelling work; this step does not select a final feature set.

In [3]:
contract_path = ROOT / 'data' / 'feature_contract.md'
handover_path = ROOT / 'docs' / 'MEMBER1_HANDOVER.md'
contract_text = contract_path.read_text(encoding='utf-8')
handover_text = handover_path.read_text(encoding='utf-8')
feature_contract_read_pass = (
    contract_path.is_file() and handover_path.is_file()
    and 'trip_duration_minutes' in contract_text
    and 'Duration' in contract_text
    and '31,988,176' in handover_text
)

safe_pretrip_candidates = [
    'provider_code', 'pickup_hour', 'day_of_week', 'month', 'weekend', 'pickup_date',
    'origin_loc_id', 'dest_loc_id', 'pickup_borough_name', 'pickup_zone_name',
    'pickup_service_zone', 'dropoff_borough_name', 'dropoff_zone_name',
    'dropoff_service_zone', 'route_id',
]
conditional_pretrip_candidates = [
    'distance_miles', 'rider_count', 'rate_class_id',
    'fare_settlement_method', 'offline_record_flag',
]
forbidden_duration_leakage = [
    'dropoff_timestamp', 'trip_duration_minutes', 'speed_mph',
    'base_fare', 'surcharge_misc', 'transit_tax', 'driver_tip_payment',
    'toll_total', 'service_improvement_fee', 'charge_total',
    'zone_congestion_fee', 'Airport_fee', 'congestion_relief_fee',
    'source_file', 'source_month', 'source_row_1based',
    'audit_zero_distance_nonzero_fare', 'audit_zero_riders',
    'audit_speed_80_to_100', 'audit_pickup_outside_source_month',
    'audit_dropoff_outside_source_month', 'audit_boundary_category',
    'audit_pickup_incomplete_zone_labels', 'audit_dropoff_incomplete_zone_labels',
]
raw_time_restriction = [
    'pickup_timestamp: use only to derive approved calendar features when request/planned time is available',
    'dropoff_timestamp: forbidden because it is observed after the trip and directly determines the target',
]

schema_columns = set(split_info['train']['columns'])
print('Feature contract and Member 1 handover read:', 'PASS' if feature_contract_read_pass else 'FAIL')
print('\nSafe pre-trip candidate columns (not a final feature set):')
print(safe_pretrip_candidates)
print('\nConditional pre-trip columns (availability must be proven at prediction time):')
print(conditional_pretrip_candidates)
print('\nForbidden duration-leakage and provenance/audit columns:')
print(forbidden_duration_leakage)
print('\nRaw timestamp rules:')
for rule in raw_time_restriction:
    print('-', rule)
print('\nLeakage rationale: speed_mph uses duration; dropoff time determines duration; completed-trip fares/charges are post-trip outcomes.')
print('Conditional columns become forbidden if they are populated only during or after trip completion.')
assert feature_contract_read_pass, 'Feature contract or Member 1 handover could not be verified.'
assert all(column in schema_columns for column in safe_pretrip_candidates + conditional_pretrip_candidates), 'A documented candidate column is absent from the split schema.' 

Feature contract and Member 1 handover read: PASS

Safe pre-trip candidate columns (not a final feature set):
['provider_code', 'pickup_hour', 'day_of_week', 'month', 'weekend', 'pickup_date', 'origin_loc_id', 'dest_loc_id', 'pickup_borough_name', 'pickup_zone_name', 'pickup_service_zone', 'dropoff_borough_name', 'dropoff_zone_name', 'dropoff_service_zone', 'route_id']

Conditional pre-trip columns (availability must be proven at prediction time):
['distance_miles', 'rider_count', 'rate_class_id', 'fare_settlement_method', 'offline_record_flag']

Forbidden duration-leakage and provenance/audit columns:
['dropoff_timestamp', 'trip_duration_minutes', 'speed_mph', 'base_fare', 'surcharge_misc', 'transit_tax', 'driver_tip_payment', 'toll_total', 'service_improvement_fee', 'charge_total', 'zone_congestion_fee', 'Airport_fee', 'congestion_relief_fee', 'source_file', 'source_month', 'source_row_1based', 'audit_zero_distance_nonzero_fare', 'audit_zero_riders', 'audit_speed_80_to_100', 'audit_p

### 4. Small train and validation target samples

Only the first 1,000 rows from the first row group of train and validation are read. These descriptive statistics confirm target availability and basic scale; they are not used to choose features, fit preprocessing, or train an estimator. No test target values are inspected.

In [4]:
sample_columns = [
    target, 'provider_code', 'pickup_hour', 'day_of_week', 'month', 'weekend',
    'origin_loc_id', 'dest_loc_id', 'distance_miles', 'rider_count',
    'rate_class_id', 'fare_settlement_method', 'offline_record_flag',
]
samples = {}
for name in ('train', 'validation'):
    parquet_file = pq.ParquetFile(split_paths[name])
    sample = parquet_file.read_row_group(0, columns=sample_columns).slice(0, 1_000).to_pandas()
    samples[name] = sample
    target_series = sample[target]
    print(f'\n{name.upper()} sample rows: {len(sample):,}')
    print(f'Target non-null: {target_series.notna().sum():,}; missing: {target_series.isna().sum():,}')
    print(target_series.describe(percentiles=[0.25, 0.5, 0.75]).to_string())
    print('Sample preview:')
    print(sample.head(5).to_string(index=False))

print('\nCandidate feature names:')
print(safe_pretrip_candidates)
print('Conditional features:')
print(conditional_pretrip_candidates)
print('Forbidden leakage fields:')
print(forbidden_duration_leakage + ['raw dropoff timestamp', 'unrestricted raw timestamps'])
print('Final feature set selected in Step 1: NO')
print('Preprocessing built in Step 1: NO')
print('Estimator trained in Step 1: NO') 


TRAIN sample rows: 1,000
Target non-null: 1,000; missing: 0
count    1000.000000
mean       13.632717
std         9.187103
min         0.050000
25%         6.712500
50%        11.391667
75%        18.700000
max        57.733333
Sample preview:
 trip_duration_minutes  provider_code  pickup_hour day_of_week  month  weekend  origin_loc_id  dest_loc_id  distance_miles  rider_count  rate_class_id  fare_settlement_method offline_record_flag
             26.316667              1            0     Tuesday      4    False            138          230            9.50          1.0            1.0                       1                   N
             10.733333              2            0     Tuesday      4    False            138           92            3.77          2.0            1.0                       1                   N
             11.083333              2            0     Tuesday      4    False            132          130            5.41          1.0            1.0                    

### 5. Step 1 verification result

This final check confirms that the split artifacts remained unchanged during the read-only audit and records the Step 1 gates.

In [5]:
source_snapshots_after = {
    name: (path.stat().st_size, path.stat().st_mtime_ns)
    for name, path in split_paths.items() if path.exists()
}
member1_data_unchanged = source_snapshots_before == source_snapshots_after
leakage_review_pass = (
    target in forbidden_duration_leakage
    and 'dropoff_timestamp' in forbidden_duration_leakage
    and 'speed_mph' in forbidden_duration_leakage
)
test_untouched_for_decisions_pass = True  # schema/row count and pickup timestamp metadata only
step1_checks = {
    'split verification': split_verification_pass,
    'chronology': chronology_pass,
    'target available': target_available_pass,
    'feature contract read': feature_contract_read_pass,
    'leakage review': leakage_review_pass,
    'test untouched for modelling decisions': test_untouched_for_decisions_pass,
    'Member 1 split files unchanged': member1_data_unchanged,
}
for check, passed in step1_checks.items():
    print(f'{check}: {"PASS" if passed else "FAIL"}')
overall_step1_pass = all(step1_checks.values())
print('Overall Step 1 status:', 'PASS' if overall_step1_pass else 'FAIL')
assert overall_step1_pass, 'One or more Step 1 verification gates failed.' 

split verification: PASS
chronology: PASS
target available: PASS
feature contract read: PASS
leakage review: PASS
test untouched for modelling decisions: PASS
Member 1 split files unchanged: PASS
Overall Step 1 status: PASS


## Step 2 — Define the initial duration feature set

The first-pass duration model uses seven compact pre-trip inputs: calendar context plus provider and origin/destination identifiers. This simple configuration creates an interpretable benchmark before testing higher-cardinality or conditional features.

`route_id` and redundant geographic text fields are deferred because they duplicate information already represented by the origin and destination IDs and can greatly expand one-hot encoding. `distance_miles` remains conditional on a route estimate being available before departure and is reserved for a later controlled experiment.

In [6]:
from collections import Counter
from pathlib import Path
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data' / 'splits').is_dir())
TRAIN_PATH = ROOT / 'data' / 'splits' / 'train.parquet'
VALIDATION_PATH = ROOT / 'data' / 'splits' / 'validation.parquet'
TEST_PATH = ROOT / 'data' / 'splits' / 'test.parquet'
TARGET = 'trip_duration_minutes'
EXPECTED_TRAIN_ROWS = 31_988_176
EXPECTED_VALIDATION_ROWS = 6_814_901

numeric_features = ['pickup_hour', 'month']
categorical_features = [
    'provider_code', 'day_of_week', 'weekend', 'origin_loc_id', 'dest_loc_id'
]
model_features = numeric_features + categorical_features

train_parquet = pq.ParquetFile(TRAIN_PATH)
validation_parquet = pq.ParquetFile(VALIDATION_PATH)
assert train_parquet.metadata.num_rows == EXPECTED_TRAIN_ROWS
assert validation_parquet.metadata.num_rows == EXPECTED_VALIDATION_ROWS
assert len(model_features) == 7
assert set(model_features + [TARGET]).issubset(train_parquet.schema_arrow.names)
assert set(model_features + [TARGET]).issubset(validation_parquet.schema_arrow.names)

split_snapshots_before = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in (TRAIN_PATH, VALIDATION_PATH, TEST_PATH)
}

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)
print('Total raw input features:', len(model_features))
print('Target:', TARGET)
print('Deferred:', ['pickup_date', 'route_id', 'zone/borough names', 'distance_miles',
                    'rider_count', 'rate_class_id', 'fare_settlement_method', 'offline_record_flag'])
print('Leakage excluded: target, dropoff_timestamp, speed_mph, post-trip fields, audit fields, and provenance fields.')
print('Step 2 status: PASS')

Numeric features: ['pickup_hour', 'month']
Categorical features: ['provider_code', 'day_of_week', 'weekend', 'origin_loc_id', 'dest_loc_id']
Total raw input features: 7
Target: trip_duration_minutes
Deferred: ['pickup_date', 'route_id', 'zone/borough names', 'distance_miles', 'rider_count', 'rate_class_id', 'fare_settlement_method', 'offline_record_flag']
Leakage excluded: target, dropoff_timestamp, speed_mph, post-trip fields, audit fields, and provenance fields.
Step 2 status: PASS


## Step 3 — Build and verify preprocessing

Numeric missing values use the training-sample median. Categorical missing values use the training-sample mode before one-hot encoding, and unseen validation categories are ignored safely. Preprocessing remains inside the eventual model `Pipeline` so the same fitted transformations are applied consistently during training and prediction.

This verification fits only the preprocessor on 1,000 training rows and transforms 1,000 train and 1,000 validation rows. It does not train a regression estimator.

In [7]:
train_small = train_parquet.read_row_group(0, columns=model_features + [TARGET]).slice(0, 1_000).to_pandas()
validation_small = validation_parquet.read_row_group(0, columns=model_features + [TARGET]).slice(0, 1_000).to_pandas()

missingness_train_small = train_small[model_features].isna().sum()
missingness_validation_small = validation_small[model_features].isna().sum()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

X_train_small = train_small[model_features]
X_validation_small = validation_small[model_features]
preprocessor.fit(X_train_small)
transformed_train_small = preprocessor.transform(X_train_small)
transformed_validation_small = preprocessor.transform(X_validation_small)

unseen_validation_categories = {}
for column in categorical_features:
    train_values = set(X_train_small[column].dropna().tolist())
    validation_values = set(X_validation_small[column].dropna().tolist())
    unseen_validation_categories[column] = sorted(validation_values - train_values, key=str)

encoder = preprocessor.named_transformers_['categorical'].named_steps['onehot']
unknown_categories_handled = (
    encoder.handle_unknown == 'ignore'
    and transformed_validation_small.shape[0] == len(validation_small)
)
transformed_feature_count = transformed_train_small.shape[1]

print('Training-sample selected-feature missingness:')
print(missingness_train_small.to_string())
print('\nValidation-sample selected-feature missingness:')
print(missingness_validation_small.to_string())
print('\nOriginal train sample shape:', X_train_small.shape)
print('Original validation sample shape:', X_validation_small.shape)
print('Transformed train shape:', transformed_train_small.shape)
print('Transformed validation shape:', transformed_validation_small.shape)
print('Transformed feature count:', transformed_feature_count)
print('Unseen validation values by categorical column:', unseen_validation_categories)
print('Validation transformation succeeds:', transformed_validation_small.shape[0] == 1_000)
print('Unknown validation categories handled:', unknown_categories_handled)
print('Regression model trained during Step 3: NO')
assert transformed_train_small.shape[0] == 1_000
assert transformed_validation_small.shape[0] == 1_000
assert transformed_train_small.shape[1] == transformed_validation_small.shape[1]
assert unknown_categories_handled
print('Step 3 status: PASS')

Training-sample selected-feature missingness:
pickup_hour      0
month            0
provider_code    0
day_of_week      0
weekend          0
origin_loc_id    0
dest_loc_id      0

Validation-sample selected-feature missingness:
pickup_hour      0
month            0
provider_code    0
day_of_week      0
weekend          0
origin_loc_id    0
dest_loc_id      0

Original train sample shape: (1000, 7)
Original validation sample shape: (1000, 7)
Transformed train shape: (1000, 228)
Transformed validation shape: (1000, 228)
Transformed feature count: 228
Unseen validation values by categorical column: {'provider_code': [], 'day_of_week': ['Thursday'], 'weekend': [], 'origin_loc_id': [106, 112, 129, 13, 160, 20, 223, 224, 243, 256, 4, 41, 69, 75, 87, 88, 97], 'dest_loc_id': [115, 120, 150, 16, 167, 173, 174, 180, 189, 40, 53, 54, 69, 78]}
Validation transformation succeeds: True
Unknown validation categories handled: True
Regression model trained during Step 3: NO
Step 3 status: PASS


## Step 4 — Median duration baseline

The benchmark predicts the exact training-split median duration for every validation record. The training target and full validation target are streamed by row group. Durations are stored as timestamp-second differences, so an integer-second frequency table gives an exact median without retaining millions of target values in memory.

Future duration models must meaningfully improve on this constant benchmark.

In [8]:
def update_second_histogram(counter, values):
    values = np.asarray(values, dtype='float64')
    assert np.isfinite(values).all(), 'Target contains non-finite values.'
    seconds = np.rint(values * 60.0).astype('int64')
    assert np.max(np.abs(values - seconds / 60.0)) < 1e-8, 'Target is not quantized to whole seconds.'
    unique_seconds, counts = np.unique(seconds, return_counts=True)
    counter.update(dict(zip(unique_seconds.tolist(), counts.tolist())))
    return len(values), float(values.sum()), float(np.square(values).sum())


def exact_histogram_median(counter, total_count):
    ranks = ((total_count - 1) // 2, total_count // 2)
    found = []
    cumulative = 0
    rank_index = 0
    for seconds, count in sorted(counter.items()):
        next_cumulative = cumulative + count
        while rank_index < len(ranks) and ranks[rank_index] < next_cumulative:
            found.append(seconds / 60.0)
            rank_index += 1
        if rank_index == len(ranks):
            break
        cumulative = next_cumulative
    assert len(found) == 2
    return (found[0] + found[1]) / 2.0


# One sequential training pass computes the exact median and the deterministic model sample.
train_duration_histogram = Counter()
train_target_rows = 0
training_sample_tables = []
SAMPLE_PER_ROW_GROUP = 2_500
for row_group_index in range(train_parquet.num_row_groups):
    table = train_parquet.read_row_group(row_group_index, columns=model_features + [TARGET])
    target_values = table[TARGET].to_numpy(zero_copy_only=False)
    rows, _, _ = update_second_histogram(train_duration_histogram, target_values)
    train_target_rows += rows
    sample_size = min(SAMPLE_PER_ROW_GROUP, table.num_rows)
    indices = np.linspace(0, table.num_rows - 1, num=sample_size, dtype=np.int64)
    training_sample_tables.append(table.take(pa.array(indices)))

training_duration_median = exact_histogram_median(train_duration_histogram, train_target_rows)
training_sample = pa.concat_tables(training_sample_tables).to_pandas()
del training_sample_tables
assert train_target_rows == EXPECTED_TRAIN_ROWS
assert len(training_sample) == 445_000

# Stream the entire validation target for baseline metrics and exact summary statistics.
validation_duration_histogram = Counter()
baseline_validation_rows = 0
baseline_absolute_error_sum = 0.0
baseline_squared_error_sum = 0.0
validation_target_sum = 0.0
validation_target_squared_sum = 0.0
for row_group_index in range(validation_parquet.num_row_groups):
    table = validation_parquet.read_row_group(row_group_index, columns=[TARGET])
    actual = table[TARGET].to_numpy(zero_copy_only=False).astype('float64', copy=False)
    errors = training_duration_median - actual
    rows, target_sum, target_squared_sum = update_second_histogram(validation_duration_histogram, actual)
    baseline_validation_rows += rows
    baseline_absolute_error_sum += float(np.abs(errors).sum())
    baseline_squared_error_sum += float(np.square(errors).sum())
    validation_target_sum += target_sum
    validation_target_squared_sum += target_squared_sum

validation_target_mean = validation_target_sum / baseline_validation_rows
validation_target_median = exact_histogram_median(validation_duration_histogram, baseline_validation_rows)
validation_total_sum_of_squares = (
    validation_target_squared_sum - validation_target_sum**2 / baseline_validation_rows
)
baseline_mae = baseline_absolute_error_sum / baseline_validation_rows
baseline_rmse = float(np.sqrt(baseline_squared_error_sum / baseline_validation_rows))
baseline_r2 = 1.0 - baseline_squared_error_sum / validation_total_sum_of_squares
assert baseline_validation_rows == EXPECTED_VALIDATION_ROWS

print('Training duration median (minutes):', f'{training_duration_median:.6f}')
print('Validation target mean (minutes):', f'{validation_target_mean:.6f}')
print('Validation target median (minutes):', f'{validation_target_median:.6f}')
print('\n| Model | Validation Rows | MAE (minutes) | RMSE (minutes) | R² |')
print('|---|---:|---:|---:|---:|')
print(f'| Median baseline | {baseline_validation_rows:,} | {baseline_mae:.6f} | {baseline_rmse:.6f} | {baseline_r2:.6f} |')
print('Interpretation: subsequent models must materially reduce MAE and RMSE and improve R² over this train-only constant benchmark.')
print('Step 4 status: PASS')

Training duration median (minutes): 13.966667
Validation target mean (minutes): 18.075826
Validation target median (minutes): 14.000000

| Model | Validation Rows | MAE (minutes) | RMSE (minutes) | R² |
|---|---:|---:|---:|---:|
| Median baseline | 6,814,901 | 9.756819 | 26.446473 | -0.024739 |
Interpretation: subsequent models must materially reduce MAE and RMSE and improve R² over this train-only constant benchmark.
Step 4 status: PASS


## Step 5 — First-pass LinearRegression

The first regression model keeps the verified preprocessing inside a single sklearn `Pipeline`. It is fitted only on the established deterministic training sample: 2,500 evenly spaced rows from each training row group, totaling 445,000 rows. Predictions and error aggregates are streamed across the entire validation split.

The test split is not opened or evaluated. This step does not tune parameters or train nonlinear candidates.

In [9]:
duration_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), numeric_features),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ])),
    ('model', LinearRegression()),
])

X_train_sample = training_sample[model_features]
y_train_sample = training_sample[TARGET].to_numpy(dtype='float64')
assert len(training_sample) == 445_000

fit_start = time.perf_counter()
duration_pipeline.fit(X_train_sample, y_train_sample)
linear_training_time_seconds = time.perf_counter() - fit_start

linear_validation_rows = 0
linear_absolute_error_sum = 0.0
linear_squared_error_sum = 0.0
linear_target_sum = 0.0
linear_target_squared_sum = 0.0
for row_group_index in range(validation_parquet.num_row_groups):
    table = validation_parquet.read_row_group(row_group_index, columns=model_features + [TARGET])
    frame = table.to_pandas()
    actual = frame[TARGET].to_numpy(dtype='float64')
    predictions = duration_pipeline.predict(frame[model_features])
    errors = predictions - actual
    linear_validation_rows += len(actual)
    linear_absolute_error_sum += float(np.abs(errors).sum())
    linear_squared_error_sum += float(np.square(errors).sum())
    linear_target_sum += float(actual.sum())
    linear_target_squared_sum += float(np.square(actual).sum())

linear_sst = linear_target_squared_sum - linear_target_sum**2 / linear_validation_rows
linear_mae = linear_absolute_error_sum / linear_validation_rows
linear_rmse = float(np.sqrt(linear_squared_error_sum / linear_validation_rows))
linear_r2 = 1.0 - linear_squared_error_sum / linear_sst
assert linear_validation_rows == EXPECTED_VALIDATION_ROWS

mae_improvement = baseline_mae - linear_mae
rmse_improvement = baseline_rmse - linear_rmse
r2_improvement = linear_r2 - baseline_r2
mae_improvement_percent = mae_improvement / baseline_mae * 100.0
rmse_improvement_percent = rmse_improvement / baseline_rmse * 100.0
linear_beats_baseline = linear_mae < baseline_mae and linear_rmse < baseline_rmse and linear_r2 > baseline_r2

print('| Model | Train Rows | Validation Rows | MAE (minutes) | RMSE (minutes) | R² | Training Time |')
print('|---|---:|---:|---:|---:|---:|---:|')
print(f'| Median baseline | {EXPECTED_TRAIN_ROWS:,} targets for median | {baseline_validation_rows:,} | {baseline_mae:.6f} | {baseline_rmse:.6f} | {baseline_r2:.6f} | n/a |')
print(f'| LinearRegression | {len(training_sample):,} | {linear_validation_rows:,} | {linear_mae:.6f} | {linear_rmse:.6f} | {linear_r2:.6f} | {linear_training_time_seconds:.3f}s |')
print('\nImprovement relative to median baseline:')
print(f'MAE improvement: {mae_improvement:.6f} minutes ({mae_improvement_percent:.3f}%)')
print(f'RMSE improvement: {rmse_improvement:.6f} minutes ({rmse_improvement_percent:.3f}%)')
print(f'R² improvement: {r2_improvement:.6f}')
print('LinearRegression beats baseline on all metrics:', linear_beats_baseline)
if linear_beats_baseline:
    print('Interpretation: LinearRegression is a useful first-pass model; a nonlinear candidate may still be worthwhile because route/time effects can be nonlinear.')
else:
    print('Interpretation: LinearRegression does not beat the constant benchmark consistently; testing a nonlinear candidate is likely worthwhile.')

split_snapshots_after = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in (TRAIN_PATH, VALIDATION_PATH, TEST_PATH)
}
member1_data_unchanged = split_snapshots_before == split_snapshots_after
assert member1_data_unchanged
print('Test split opened during Steps 2–5: NO')
print('Member 1 split files unchanged: PASS')
print('Step 5 status: PASS')
print('Overall Steps 2–5 status: PASS')
print('STOP: no nonlinear model, conditional distance experiment, tuning, test evaluation, plots, or serialization performed.')

| Model | Train Rows | Validation Rows | MAE (minutes) | RMSE (minutes) | R² | Training Time |
|---|---:|---:|---:|---:|---:|---:|
| Median baseline | 31,988,176 targets for median | 6,814,901 | 9.756819 | 26.446473 | -0.024739 | n/a |
| LinearRegression | 445,000 | 6,814,901 | 8.566630 | 24.971912 | 0.086347 | 3.630s |

Improvement relative to median baseline:
MAE improvement: 1.190189 minutes (12.199%)
RMSE improvement: 1.474561 minutes (5.576%)
R² improvement: 0.111086
LinearRegression beats baseline on all metrics: True
Interpretation: LinearRegression is a useful first-pass model; a nonlinear candidate may still be worthwhile because route/time effects can be nonlinear.
Test split opened during Steps 2–5: NO
Member 1 split files unchanged: PASS
Step 5 status: PASS
Overall Steps 2–5 status: PASS
STOP: no nonlinear model, conditional distance experiment, tuning, test evaluation, plots, or serialization performed.


## Steps 2–5 final summary

### Step 2 — PASS

- Numeric features: `pickup_hour`, `month`
- Categorical features: `provider_code`, `day_of_week`, `weekend`, `origin_loc_id`, `dest_loc_id`
- Total raw input features: **7**

### Step 3 — PASS

- Missingness in the selected features: **none in either 1,000-row sample**
- Raw train sample shape: **(1,000, 7)**
- Raw validation sample shape: **(1,000, 7)**
- Transformed train shape: **(1,000, 228)**
- Transformed validation shape: **(1,000, 228)**
- Transformed feature count: **228**
- Validation transformation and unknown-category handling: **PASS**

### Step 4 — PASS

- Training duration median: **13.966667 minutes**
- Validation target mean: **18.075826 minutes**
- Validation target median: **14.000000 minutes**
- Validation MAE: **9.756819 minutes**
- Validation RMSE: **26.446473 minutes**
- Validation R²: **−0.024739**
- Validation rows evaluated: **6,814,901**

### Step 5 — PASS

- Training rows used: **445,000**
- Validation rows evaluated: **6,814,901**
- Training time: **3.630 seconds**
- Validation MAE: **8.566630 minutes**
- Validation RMSE: **24.971912 minutes**
- Validation R²: **0.086347**
- MAE improvement over baseline: **1.190189 minutes (12.199%)**
- RMSE improvement over baseline: **1.474561 minutes (5.576%)**
- R² improvement over baseline: **0.111086**

LinearRegression beats the median baseline on all three metrics. Its modest R² indicates that nonlinear candidates may be worth testing in a later authorized step. Long-duration observations were retained unchanged. The test split was not opened or used during Steps 2–5.

**Overall Steps 2–5 status: PASS**


## Step 6 — RandomForestRegressor

This fixed first-pass Random Forest uses the same seven inputs, preprocessing contract, and deterministic 445,000-row training sample as LinearRegression. It is evaluated against the entire validation split without tuning or test access.

In [10]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data' / 'splits').is_dir())
TRAIN_PATH = ROOT / 'data' / 'splits' / 'train.parquet'
VALIDATION_PATH = ROOT / 'data' / 'splits' / 'validation.parquet'
CONTRACT_PATH = ROOT / 'data' / 'feature_contract.md'
TARGET = 'trip_duration_minutes'
EXPECTED_TRAIN_ROWS = 31_988_176
EXPECTED_VALIDATION_ROWS = 6_814_901
EXPECTED_SAMPLE_ROWS = 445_000
SAMPLE_PER_ROW_GROUP = 2_500

numeric_features = ['pickup_hour', 'month']
categorical_features = ['provider_code', 'day_of_week', 'weekend', 'origin_loc_id', 'dest_loc_id']
model_features = numeric_features + categorical_features
sample_columns = model_features + ['distance_miles', 'route_id', TARGET]

train_parquet = pq.ParquetFile(TRAIN_PATH)
assert train_parquet.metadata.num_rows == EXPECTED_TRAIN_ROWS
assert set(sample_columns).issubset(train_parquet.schema_arrow.names)
split_snapshots_before_6_9 = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in (TRAIN_PATH, VALIDATION_PATH)
}

# Exact established sample: 2,500 evenly spaced rows per each of 178 train row groups.
sample_tables = []
for row_group_index in range(train_parquet.num_row_groups):
    table = train_parquet.read_row_group(row_group_index, columns=sample_columns)
    sample_size = min(SAMPLE_PER_ROW_GROUP, table.num_rows)
    indices = np.linspace(0, table.num_rows - 1, num=sample_size, dtype=np.int64)
    sample_tables.append(table.take(pa.array(indices)))
training_sample_6_9 = pa.concat_tables(sample_tables).to_pandas()
del sample_tables
assert train_parquet.num_row_groups == 178
assert len(training_sample_6_9) == EXPECTED_SAMPLE_ROWS


def build_preprocessor(numeric, categorical):
    return ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), numeric),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical),
    ])


def stream_validation_metrics(fitted_pipeline, features):
    validation_parquet = pq.ParquetFile(VALIDATION_PATH)
    assert validation_parquet.metadata.num_rows == EXPECTED_VALIDATION_ROWS
    assert set(features + [TARGET]).issubset(validation_parquet.schema_arrow.names)
    rows = 0
    absolute_error_sum = 0.0
    squared_error_sum = 0.0
    target_sum = 0.0
    target_squared_sum = 0.0
    for row_group_index in range(validation_parquet.num_row_groups):
        frame = validation_parquet.read_row_group(
            row_group_index, columns=features + [TARGET]
        ).to_pandas()
        actual = frame[TARGET].to_numpy(dtype='float64')
        predictions = fitted_pipeline.predict(frame[features])
        errors = predictions - actual
        rows += len(actual)
        absolute_error_sum += float(np.abs(errors).sum())
        squared_error_sum += float(np.square(errors).sum())
        target_sum += float(actual.sum())
        target_squared_sum += float(np.square(actual).sum())
    validation_parquet.close()
    assert rows == EXPECTED_VALIDATION_ROWS
    total_sum_of_squares = target_squared_sum - target_sum**2 / rows
    return {
        'rows': rows,
        'mae': absolute_error_sum / rows,
        'rmse': float(np.sqrt(squared_error_sum / rows)),
        'r2': 1.0 - squared_error_sum / total_sum_of_squares,
    }


baseline_metrics = {'mae': 9.756819, 'rmse': 26.446473, 'r2': -0.024739}
linear_metrics = {'mae': 8.566630, 'rmse': 24.971912, 'r2': 0.086347}
linear_training_time = 3.630

random_forest_config = {
    'n_estimators': 100,
    'max_depth': 18,
    'min_samples_leaf': 5,
    'max_features': 0.7,
    'random_state': 42,
    'n_jobs': -1,
}
random_forest_pipeline = Pipeline([
    ('preprocessor', build_preprocessor(numeric_features, categorical_features)),
    ('model', RandomForestRegressor(**random_forest_config)),
])
rf_fit_start = time.perf_counter()
random_forest_pipeline.fit(
    training_sample_6_9[model_features],
    training_sample_6_9[TARGET].to_numpy(dtype='float64'),
)
rf_training_time = time.perf_counter() - rf_fit_start
rf_metrics = stream_validation_metrics(random_forest_pipeline, model_features)

rf_mae_change_vs_linear = rf_metrics['mae'] - linear_metrics['mae']
rf_rmse_change_vs_linear = rf_metrics['rmse'] - linear_metrics['rmse']
rf_r2_change_vs_linear = rf_metrics['r2'] - linear_metrics['r2']
rf_beats_linear = (
    rf_metrics['mae'] < linear_metrics['mae']
    and rf_metrics['rmse'] < linear_metrics['rmse']
    and rf_metrics['r2'] > linear_metrics['r2']
)

print('RandomForest configuration:', random_forest_config)
print('Training rows:', len(training_sample_6_9))
print('Validation rows:', rf_metrics['rows'])
print('Training time (seconds):', f'{rf_training_time:.3f}')
print('Validation MAE (minutes):', f"{rf_metrics['mae']:.6f}")
print('Validation RMSE (minutes):', f"{rf_metrics['rmse']:.6f}")
print('Validation R²:', f"{rf_metrics['r2']:.6f}")
print('RandomForest minus LinearRegression MAE:', f'{rf_mae_change_vs_linear:+.6f}')
print('RandomForest minus LinearRegression RMSE:', f'{rf_rmse_change_vs_linear:+.6f}')
print('RandomForest minus LinearRegression R²:', f'{rf_r2_change_vs_linear:+.6f}')
print('RandomForest beats LinearRegression on all metrics:', rf_beats_linear)
print('Added complexity justified:', rf_beats_linear)
print('Step 6 status: PASS')

RandomForest configuration: {'n_estimators': 100, 'max_depth': 18, 'min_samples_leaf': 5, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}
Training rows: 445000
Validation rows: 6814901
Training time (seconds): 152.079
Validation MAE (minutes): 8.708620
Validation RMSE (minutes): 25.046847
Validation R²: 0.080855
RandomForest minus LinearRegression MAE: +0.141990
RandomForest minus LinearRegression RMSE: +0.074935
RandomForest minus LinearRegression R²: -0.005492
RandomForest beats LinearRegression on all metrics: False
Added complexity justified: False
Step 6 status: PASS


## Step 7 — ExtraTreesRegressor

ExtraTreesRegressor is compatible with the existing one-hot preprocessing and provides a second nonlinear comparison under the same fixed tree limits, deterministic sample, and full validation evaluation. No hyperparameters are tuned.

In [11]:
extra_trees_config = {
    'n_estimators': 100,
    'max_depth': 18,
    'min_samples_leaf': 5,
    'max_features': 0.7,
    'random_state': 42,
    'n_jobs': -1,
}
extra_trees_pipeline = Pipeline([
    ('preprocessor', build_preprocessor(numeric_features, categorical_features)),
    ('model', ExtraTreesRegressor(**extra_trees_config)),
])
et_fit_start = time.perf_counter()
extra_trees_pipeline.fit(
    training_sample_6_9[model_features],
    training_sample_6_9[TARGET].to_numpy(dtype='float64'),
)
et_training_time = time.perf_counter() - et_fit_start
et_metrics = stream_validation_metrics(extra_trees_pipeline, model_features)

et_mae_change_vs_linear = et_metrics['mae'] - linear_metrics['mae']
et_rmse_change_vs_linear = et_metrics['rmse'] - linear_metrics['rmse']
et_r2_change_vs_linear = et_metrics['r2'] - linear_metrics['r2']
et_beats_linear = (
    et_metrics['mae'] < linear_metrics['mae']
    and et_metrics['rmse'] < linear_metrics['rmse']
    and et_metrics['r2'] > linear_metrics['r2']
)

comparison_7 = [
    ('Median baseline', '31,988,176 targets for median', EXPECTED_VALIDATION_ROWS,
     baseline_metrics['mae'], baseline_metrics['rmse'], baseline_metrics['r2'], 'n/a'),
    ('LinearRegression', f'{EXPECTED_SAMPLE_ROWS:,}', EXPECTED_VALIDATION_ROWS,
     linear_metrics['mae'], linear_metrics['rmse'], linear_metrics['r2'], f'{linear_training_time:.3f}s'),
    ('RandomForestRegressor', f'{EXPECTED_SAMPLE_ROWS:,}', rf_metrics['rows'],
     rf_metrics['mae'], rf_metrics['rmse'], rf_metrics['r2'], f'{rf_training_time:.3f}s'),
    ('ExtraTreesRegressor', f'{EXPECTED_SAMPLE_ROWS:,}', et_metrics['rows'],
     et_metrics['mae'], et_metrics['rmse'], et_metrics['r2'], f'{et_training_time:.3f}s'),
]
print('ExtraTrees configuration:', extra_trees_config)
print('| Model | Train Rows | Validation Rows | MAE (minutes) | RMSE (minutes) | R² | Training Time |')
print('|---|---:|---:|---:|---:|---:|---:|')
for model_name, train_rows, validation_rows, mae, rmse, r2, training_time in comparison_7:
    print(f'| {model_name} | {train_rows} | {validation_rows:,} | {mae:.6f} | {rmse:.6f} | {r2:.6f} | {training_time} |')
print('\nExtraTrees minus LinearRegression MAE:', f'{et_mae_change_vs_linear:+.6f}')
print('ExtraTrees minus LinearRegression RMSE:', f'{et_rmse_change_vs_linear:+.6f}')
print('ExtraTrees minus LinearRegression R²:', f'{et_r2_change_vs_linear:+.6f}')
print('ExtraTrees beats LinearRegression on all metrics:', et_beats_linear)
print('Step 7 status: PASS')

ExtraTrees configuration: {'n_estimators': 100, 'max_depth': 18, 'min_samples_leaf': 5, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}
| Model | Train Rows | Validation Rows | MAE (minutes) | RMSE (minutes) | R² | Training Time |
|---|---:|---:|---:|---:|---:|---:|
| Median baseline | 31,988,176 targets for median | 6,814,901 | 9.756819 | 26.446473 | -0.024739 | n/a |
| LinearRegression | 445,000 | 6,814,901 | 8.566630 | 24.971912 | 0.086347 | 3.630s |
| RandomForestRegressor | 445,000 | 6,814,901 | 8.708620 | 25.046847 | 0.080855 | 152.079s |
| ExtraTreesRegressor | 445,000 | 6,814,901 | 8.696595 | 25.090268 | 0.077666 | 144.491s |

ExtraTrees minus LinearRegression MAE: +0.129965
ExtraTrees minus LinearRegression RMSE: +0.118356
ExtraTrees minus LinearRegression R²: -0.008681
ExtraTrees beats LinearRegression on all metrics: False
Step 7 status: PASS


## Step 8 — Controlled `route_id` experiment

This experiment adds only `route_id` to the original seven inputs. The route-inclusive preprocessor is fitted on the same deterministic training sample, then its encoded width is checked against the predefined 5,000-feature safety limit. If the limit is exceeded, model fitting stops without introducing a different encoder.

In [12]:
route_numeric_features = ['pickup_hour', 'month']
route_categorical_features = categorical_features + ['route_id']
route_features = route_numeric_features + route_categorical_features
ROUTE_FEATURE_LIMIT = 5_000

route_preprocessor = build_preprocessor(route_numeric_features, route_categorical_features)
route_preprocessor.fit(training_sample_6_9[route_features], training_sample_6_9[TARGET])
route_encoded_feature_count = route_preprocessor.transform(
    training_sample_6_9[route_features].iloc[:1]
).shape[1]
route_fit_performed = route_encoded_feature_count <= ROUTE_FEATURE_LIMIT
route_metrics = None
route_training_time = None

print('Route-inclusive encoded feature count:', route_encoded_feature_count)
print('Safety threshold:', ROUTE_FEATURE_LIMIT)
if not route_fit_performed:
    route_decision = 'Exclude route_id under the current one-hot setup; encoded width exceeds the safety threshold.'
    print('LinearRegression fit performed:', False)
    print('Decision:', route_decision)
else:
    route_pipeline = Pipeline([
        ('preprocessor', route_preprocessor),
        ('model', LinearRegression()),
    ])
    route_fit_start = time.perf_counter()
    route_pipeline.fit(training_sample_6_9[route_features], training_sample_6_9[TARGET])
    route_training_time = time.perf_counter() - route_fit_start
    route_metrics = stream_validation_metrics(route_pipeline, route_features)
    route_decision = 'Route experiment completed because encoded width stayed within the safety threshold.'
    print('LinearRegression fit performed:', True)
    print('Validation MAE:', f"{route_metrics['mae']:.6f}")
    print('Validation RMSE:', f"{route_metrics['rmse']:.6f}")
    print('Validation R²:', f"{route_metrics['r2']:.6f}")
    print('Decision:', route_decision)
print('Step 8 status: PASS')

Route-inclusive encoded feature count: 22392
Safety threshold: 5000
LinearRegression fit performed: False
Decision: Exclude route_id under the current one-hot setup; encoded width exceeds the safety threshold.
Step 8 status: PASS


## Step 9 — Conditional `distance_miles` experiment

**This experiment assumes an estimated route distance is available at prediction time before the trip begins.**

The stored completed-trip distance must not be presented as information known before pickup. This controlled experiment adds only `distance_miles` to the first-pass LinearRegression configuration and interprets the result conditionally.

In [13]:
contract_text = CONTRACT_PATH.read_text(encoding='utf-8')
conditional_section = contract_text.split('### Conditional features', 1)[1].split('## C.', 1)[0]
distance_contract_verified = (
    '`distance_miles`' in conditional_section
    and 'before departure' in conditional_section
)
assert distance_contract_verified

distance_numeric_features = ['pickup_hour', 'month', 'distance_miles']
distance_categorical_features = categorical_features.copy()
distance_features = distance_numeric_features + distance_categorical_features
assert set(distance_features) - set(model_features) == {'distance_miles'}

distance_pipeline = Pipeline([
    ('preprocessor', build_preprocessor(distance_numeric_features, distance_categorical_features)),
    ('model', LinearRegression()),
])
distance_fit_start = time.perf_counter()
distance_pipeline.fit(
    training_sample_6_9[distance_features],
    training_sample_6_9[TARGET].to_numpy(dtype='float64'),
)
distance_training_time = time.perf_counter() - distance_fit_start
distance_metrics = stream_validation_metrics(distance_pipeline, distance_features)

distance_mae_improvement = linear_metrics['mae'] - distance_metrics['mae']
distance_rmse_improvement = linear_metrics['rmse'] - distance_metrics['rmse']
distance_mae_improvement_percent = distance_mae_improvement / linear_metrics['mae'] * 100.0
distance_rmse_improvement_percent = distance_rmse_improvement / linear_metrics['rmse'] * 100.0
distance_r2_change = distance_metrics['r2'] - linear_metrics['r2']
distance_meaningful = (
    distance_mae_improvement_percent >= 1.0
    and distance_rmse_improvement_percent >= 1.0
    and distance_r2_change > 0
)

print('Feature-contract conditional permission verified:', distance_contract_verified)
print('This experiment assumes an estimated route distance is available at prediction time before the trip begins.')
print('| Feature set | Train Rows | Validation Rows | MAE (minutes) | RMSE (minutes) | R² | Training Time |')
print('|---|---:|---:|---:|---:|---:|---:|')
print(f"| Original LinearRegression | {EXPECTED_SAMPLE_ROWS:,} | {EXPECTED_VALIDATION_ROWS:,} | {linear_metrics['mae']:.6f} | {linear_metrics['rmse']:.6f} | {linear_metrics['r2']:.6f} | {linear_training_time:.3f}s |")
print(f"| Conditional + distance_miles | {len(training_sample_6_9):,} | {distance_metrics['rows']:,} | {distance_metrics['mae']:.6f} | {distance_metrics['rmse']:.6f} | {distance_metrics['r2']:.6f} | {distance_training_time:.3f}s |")
print('\nAbsolute MAE improvement:', f'{distance_mae_improvement:.6f} minutes')
print('Percentage MAE improvement:', f'{distance_mae_improvement_percent:.3f}%')
print('Absolute RMSE improvement:', f'{distance_rmse_improvement:.6f} minutes')
print('Percentage RMSE improvement:', f'{distance_rmse_improvement_percent:.3f}%')
print('R² change:', f'{distance_r2_change:+.6f}')
print('Meaningful conditional improvement:', distance_meaningful)
print('Completed-trip mileage claimed as pre-trip information: NO')
print('Step 9 status: PASS')

Feature-contract conditional permission verified: True
This experiment assumes an estimated route distance is available at prediction time before the trip begins.
| Feature set | Train Rows | Validation Rows | MAE (minutes) | RMSE (minutes) | R² | Training Time |
|---|---:|---:|---:|---:|---:|---:|
| Original LinearRegression | 445,000 | 6,814,901 | 8.566630 | 24.971912 | 0.086347 | 3.630s |
| Conditional + distance_miles | 445,000 | 6,814,901 | 6.367238 | 23.700579 | 0.177008 | 4.286s |

Absolute MAE improvement: 2.199392 minutes
Percentage MAE improvement: 25.674%
Absolute RMSE improvement: 1.271333 minutes
Percentage RMSE improvement: 5.091%
R² change: +0.090661
Meaningful conditional improvement: True
Completed-trip mileage claimed as pre-trip information: NO
Step 9 status: PASS


## Steps 6–9 final summary

The following executed summary reports the fixed model configurations, validation comparisons, guarded route experiment, conditional distance result, and data-safety checks. No final model is selected in these steps.

In [14]:
split_snapshots_after_6_9 = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in (TRAIN_PATH, VALIDATION_PATH)
}
member1_data_unchanged_6_9 = split_snapshots_before_6_9 == split_snapshots_after_6_9
assert member1_data_unchanged_6_9

print('STEP 6 — PASS')
print('Configuration:', random_forest_config)
print(f"Rows: train={EXPECTED_SAMPLE_ROWS:,}, validation={rf_metrics['rows']:,}")
print(f"Training time={rf_training_time:.3f}s, MAE={rf_metrics['mae']:.6f}, RMSE={rf_metrics['rmse']:.6f}, R²={rf_metrics['r2']:.6f}")
print(f"Versus LinearRegression: MAE {rf_mae_change_vs_linear:+.6f}, RMSE {rf_rmse_change_vs_linear:+.6f}, R² {rf_r2_change_vs_linear:+.6f}")

print('\nSTEP 7 — PASS')
print('Model: ExtraTreesRegressor; configuration:', extra_trees_config)
print(f"Rows: train={EXPECTED_SAMPLE_ROWS:,}, validation={et_metrics['rows']:,}")
print(f"Training time={et_training_time:.3f}s, MAE={et_metrics['mae']:.6f}, RMSE={et_metrics['rmse']:.6f}, R²={et_metrics['r2']:.6f}")
print(f"Versus LinearRegression: MAE {et_mae_change_vs_linear:+.6f}, RMSE {et_rmse_change_vs_linear:+.6f}, R² {et_r2_change_vs_linear:+.6f}")

print('\nSTEP 8 — PASS')
print(f'Encoded features={route_encoded_feature_count:,}; threshold={ROUTE_FEATURE_LIMIT:,}; fit performed={route_fit_performed}')
print('Decision:', route_decision)

print('\nSTEP 9 — PASS')
print(f"Rows: train={EXPECTED_SAMPLE_ROWS:,}, validation={distance_metrics['rows']:,}")
print(f"Training time={distance_training_time:.3f}s, MAE={distance_metrics['mae']:.6f}, RMSE={distance_metrics['rmse']:.6f}, R²={distance_metrics['r2']:.6f}")
print(f'MAE improvement={distance_mae_improvement:.6f} ({distance_mae_improvement_percent:.3f}%)')
print(f'RMSE improvement={distance_rmse_improvement:.6f} ({distance_rmse_improvement_percent:.3f}%)')
print(f'R² change={distance_r2_change:+.6f}')
print('Conditional assumption: estimated route distance must be available before trip start.')

print('\nTest split accessed during Steps 6–9: NO')
print('Member 1 train/validation files unchanged: PASS')
print('Overall Steps 6–9: PASS')
print('STOP: no tuning, final selection, test evaluation, plots, combined retraining, or serialization performed.')

STEP 6 — PASS
Configuration: {'n_estimators': 100, 'max_depth': 18, 'min_samples_leaf': 5, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}
Rows: train=445,000, validation=6,814,901
Training time=152.079s, MAE=8.708620, RMSE=25.046847, R²=0.080855
Versus LinearRegression: MAE +0.141990, RMSE +0.074935, R² -0.005492

STEP 7 — PASS
Model: ExtraTreesRegressor; configuration: {'n_estimators': 100, 'max_depth': 18, 'min_samples_leaf': 5, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}
Rows: train=445,000, validation=6,814,901
Training time=144.491s, MAE=8.696595, RMSE=25.090268, R²=0.077666
Versus LinearRegression: MAE +0.129965, RMSE +0.118356, R² -0.008681

STEP 8 — PASS
Encoded features=22,392; threshold=5,000; fit performed=False
Decision: Exclude route_id under the current one-hot setup; encoded width exceeds the safety threshold.

STEP 9 — PASS
Rows: train=445,000, validation=6,814,901
Training time=4.286s, MAE=6.367238, RMSE=23.700579, R²=0.177008
MAE improvement=2.19939

## Step 10 — Final validation selection and error analysis

The validation comparison supports two defensible operating configurations:

- **Strict pre-trip configuration:** LinearRegression without `distance_miles`, for cases where no route-distance estimate is available.
- **Route-estimate configuration:** LinearRegression with `distance_miles`, only when that value is an estimated route distance available before departure.

The route-estimate configuration is selected for final evaluation because it has the best validation MAE, RMSE, and R², conditional on accepting the distance-availability assumption. Completed-trip mileage is not treated as known before pickup.

In [15]:
from collections import Counter
from pathlib import Path
import os
import tempfile
import time

import joblib
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'urban-flow-matplotlib'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data' / 'splits').is_dir())
TRAIN_PATH = ROOT / 'data' / 'splits' / 'train.parquet'
VALIDATION_PATH = ROOT / 'data' / 'splits' / 'validation.parquet'
TEST_PATH = ROOT / 'data' / 'splits' / 'test.parquet'
FIGURE_DIR = ROOT / 'reports' / 'figures'
MODEL_DIR = ROOT / 'models'
DOCS_DIR = ROOT / 'docs'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'trip_duration_minutes'
NUMERIC_FEATURES = ['pickup_hour', 'month', 'distance_miles']
CATEGORICAL_FEATURES = ['provider_code', 'day_of_week', 'weekend', 'origin_loc_id', 'dest_loc_id']
FINAL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
EXPECTED_TRAIN_ROWS = 31_988_176
EXPECTED_VALIDATION_ROWS = 6_814_901
EXPECTED_TEST_ROWS = 6_730_257
SAMPLE_PER_ROW_GROUP = 2_500

validation_comparison = [
    ('Median baseline', 9.756819, 26.446473, -0.024739, 'n/a'),
    ('LinearRegression without distance', 8.566630, 24.971912, 0.086347, '3.630s'),
    ('RandomForestRegressor', 8.708620, 25.046847, 0.080855, '152.079s'),
    ('ExtraTreesRegressor', 8.696595, 25.090268, 0.077666, '144.491s'),
    ('LinearRegression + conditional distance_miles', 6.367238, 23.700579, 0.177008, '4.286s'),
]
print('| Model | MAE (minutes) | RMSE (minutes) | R² | Training Time |')
print('|---|---:|---:|---:|---:|')
for model_name, mae, rmse, r2, training_time in validation_comparison:
    print(f'| {model_name} | {mae:.6f} | {rmse:.6f} | {r2:.6f} | {training_time} |')
print('\nSelected model: LinearRegression with conditional distance_miles')
print('Strict pre-trip fallback: LinearRegression without distance_miles')
print('Selection condition: an estimated route distance is available before departure.')


def make_preprocessor():
    return ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), NUMERIC_FEATURES),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), CATEGORICAL_FEATURES),
    ])


def histogram_percentile(counter, total, percentile):
    rank = (percentile / 100.0) * (total - 1)
    lower_rank = int(np.floor(rank))
    upper_rank = int(np.ceil(rank))
    wanted = [lower_rank, upper_rank]
    found = []
    cumulative = 0
    wanted_index = 0
    for seconds, count in sorted(counter.items()):
        next_cumulative = cumulative + count
        while wanted_index < len(wanted) and wanted[wanted_index] < next_cumulative:
            found.append(seconds / 60.0)
            wanted_index += 1
        if wanted_index == len(wanted):
            break
        cumulative = next_cumulative
    assert len(found) == 2
    fraction = rank - lower_rank
    return found[0] + fraction * (found[1] - found[0])


# Recreate the exact 445,000-row deterministic training sample used in Steps 5–9.
train_parquet_10 = pq.ParquetFile(TRAIN_PATH)
assert train_parquet_10.metadata.num_rows == EXPECTED_TRAIN_ROWS
step10_sample_tables = []
for row_group_index in range(train_parquet_10.num_row_groups):
    table = train_parquet_10.read_row_group(row_group_index, columns=FINAL_FEATURES + [TARGET])
    sample_size = min(SAMPLE_PER_ROW_GROUP, table.num_rows)
    indices = np.linspace(0, table.num_rows - 1, num=sample_size, dtype=np.int64)
    step10_sample_tables.append(table.take(pa.array(indices)))
step10_training_sample = pa.concat_tables(step10_sample_tables).to_pandas()
del step10_sample_tables
train_parquet_10.close()
assert len(step10_training_sample) == 445_000

selected_validation_pipeline = Pipeline([
    ('preprocessor', make_preprocessor()),
    ('model', LinearRegression()),
])
step10_fit_start = time.perf_counter()
selected_validation_pipeline.fit(
    step10_training_sample[FINAL_FEATURES],
    step10_training_sample[TARGET].to_numpy(dtype='float64'),
)
step10_refit_time = time.perf_counter() - step10_fit_start

# Full-validation metrics, segment errors, duration distribution, and a bounded plotting sample.
validation_parquet_10 = pq.ParquetFile(VALIDATION_PATH)
assert validation_parquet_10.metadata.num_rows == EXPECTED_VALIDATION_ROWS
validation_rows_10 = 0
absolute_error_sum_10 = 0.0
squared_error_sum_10 = 0.0
target_sum_10 = 0.0
target_squared_sum_10 = 0.0
hour_abs_error = np.zeros(24, dtype='float64')
hour_counts = np.zeros(24, dtype='int64')
distance_edges = np.array([0.0, 2.0, 5.0, 10.0, 20.0, np.inf])
distance_labels = ['0–2 miles', '2–5 miles', '5–10 miles', '10–20 miles', '20+ miles']
distance_abs_error = np.zeros(len(distance_labels), dtype='float64')
distance_counts = np.zeros(len(distance_labels), dtype='int64')
duration_histogram = Counter()
very_long_threshold_minutes = 120.0
very_long_count = 0
plot_actual_parts, plot_prediction_parts = [], []
plot_residual_parts = []

for row_group_index in range(validation_parquet_10.num_row_groups):
    frame = validation_parquet_10.read_row_group(
        row_group_index, columns=FINAL_FEATURES + [TARGET]
    ).to_pandas()
    actual = frame[TARGET].to_numpy(dtype='float64')
    prediction = selected_validation_pipeline.predict(frame[FINAL_FEATURES])
    residual = actual - prediction
    absolute_error = np.abs(residual)
    validation_rows_10 += len(actual)
    absolute_error_sum_10 += float(absolute_error.sum())
    squared_error_sum_10 += float(np.square(residual).sum())
    target_sum_10 += float(actual.sum())
    target_squared_sum_10 += float(np.square(actual).sum())

    hours = frame['pickup_hour'].to_numpy(dtype='int64')
    hour_counts += np.bincount(hours, minlength=24)[:24]
    hour_abs_error += np.bincount(hours, weights=absolute_error, minlength=24)[:24]

    distances = frame['distance_miles'].to_numpy(dtype='float64')
    bucket_indices = np.searchsorted(distance_edges, distances, side='right') - 1
    valid_buckets = np.isfinite(distances) & (bucket_indices >= 0) & (bucket_indices < len(distance_labels))
    distance_counts += np.bincount(bucket_indices[valid_buckets], minlength=len(distance_labels))[:len(distance_labels)]
    distance_abs_error += np.bincount(
        bucket_indices[valid_buckets], weights=absolute_error[valid_buckets], minlength=len(distance_labels)
    )[:len(distance_labels)]

    seconds = np.rint(actual * 60.0).astype('int64')
    unique_seconds, counts = np.unique(seconds, return_counts=True)
    duration_histogram.update(dict(zip(unique_seconds.tolist(), counts.tolist())))
    very_long_count += int((actual > very_long_threshold_minutes).sum())

    plot_size = min(1_000, len(actual))
    plot_indices = np.linspace(0, len(actual) - 1, num=plot_size, dtype=np.int64)
    plot_actual_parts.append(actual[plot_indices])
    plot_prediction_parts.append(prediction[plot_indices])
    plot_residual_parts.append(residual[plot_indices])

validation_parquet_10.close()
assert validation_rows_10 == EXPECTED_VALIDATION_ROWS
validation_sst_10 = target_squared_sum_10 - target_sum_10**2 / validation_rows_10
selected_validation_metrics = {
    'mae': absolute_error_sum_10 / validation_rows_10,
    'rmse': float(np.sqrt(squared_error_sum_10 / validation_rows_10)),
    'r2': 1.0 - squared_error_sum_10 / validation_sst_10,
    'rows': validation_rows_10,
}
assert abs(selected_validation_metrics['mae'] - 6.367238) < 0.00001
assert abs(selected_validation_metrics['rmse'] - 23.700579) < 0.00001
assert abs(selected_validation_metrics['r2'] - 0.177008) < 0.00001

hour_mae = np.divide(hour_abs_error, hour_counts, out=np.full(24, np.nan), where=hour_counts > 0)
distance_mae = np.divide(
    distance_abs_error, distance_counts, out=np.full(len(distance_labels), np.nan), where=distance_counts > 0
)
strongest_hour = int(np.nanargmin(hour_mae))
weakest_hour = int(np.nanargmax(hour_mae))
strongest_distance_bucket = distance_labels[int(np.nanargmin(distance_mae))]
weakest_distance_bucket = distance_labels[int(np.nanargmax(distance_mae))]
duration_percentiles = {
    percentile: histogram_percentile(duration_histogram, validation_rows_10, percentile)
    for percentile in [50, 90, 95, 99, 99.5, 99.9]
}
maximum_validation_duration = max(duration_histogram) / 60.0
very_long_percentage = very_long_count / validation_rows_10 * 100.0

plot_actual = np.concatenate(plot_actual_parts)
plot_prediction = np.concatenate(plot_prediction_parts)
plot_residual = np.concatenate(plot_residual_parts)
del plot_actual_parts, plot_prediction_parts, plot_residual_parts

plt.style.use('seaborn-v0_8-whitegrid')

# Predicted versus actual: central display range only; all rows remain in reported metrics.
actual_limit = float(np.percentile(plot_actual, 99.5))
prediction_low, prediction_high = np.percentile(plot_prediction, [0.5, 99.5])
fig, ax = plt.subplots(figsize=(8, 6))
hexbin = ax.hexbin(plot_actual, plot_prediction, gridsize=55, mincnt=1, cmap='viridis')
line_low = min(0.0, prediction_low)
line_high = max(actual_limit, prediction_high)
ax.plot([line_low, line_high], [line_low, line_high], '--', color='crimson', linewidth=1.5, label='Ideal prediction')
ax.set_xlim(0, actual_limit)
ax.set_ylim(prediction_low, prediction_high)
ax.set_title('Duration Predictions vs Actuals\nValidation sample; central 99.5% display range')
ax.set_xlabel('Actual trip duration (minutes)')
ax.set_ylabel('Predicted trip duration (minutes)')
ax.legend()
fig.colorbar(hexbin, ax=ax, label='Sample density')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'duration_predicted_vs_actual.png', dpi=180, bbox_inches='tight')
plt.close(fig)

# Residual display uses a central range for readability; metrics and analysis retain all observations.
residual_low, residual_high = np.percentile(plot_residual, [0.5, 99.5])
fig, ax = plt.subplots(figsize=(8, 5))
central_residuals = plot_residual[(plot_residual >= residual_low) & (plot_residual <= residual_high)]
ax.hist(central_residuals, bins=80, color='#4472C4', alpha=0.85)
ax.axvline(0, color='crimson', linestyle='--', linewidth=1.5)
ax.set_title('Duration Residual Distribution\nValidation sample; central 99% display range')
ax.set_xlabel('Residual: actual − predicted (minutes)')
ax.set_ylabel('Trips in plotting sample')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'duration_residual_distribution.png', dpi=180, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(np.arange(24), hour_mae, color='#2F75B5')
ax.set_title('Validation MAE by Pickup Hour')
ax.set_xlabel('Pickup hour (0–23)')
ax.set_ylabel('Mean absolute error (minutes)')
ax.set_xticks(np.arange(24))
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'duration_mae_by_hour.png', dpi=180, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(distance_labels, distance_mae, color='#70AD47')
ax.set_title('Validation MAE by Estimated Distance Bucket')
ax.set_xlabel('Estimated route distance (miles)')
ax.set_ylabel('Mean absolute error (minutes)')
ax.tick_params(axis='x', rotation=15)
for bar, value in zip(bars, distance_mae):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.2f}', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'duration_mae_by_distance_bucket.png', dpi=180, bbox_inches='tight')
plt.close(fig)

print(f'Validation model refit time for error analysis: {step10_refit_time:.3f}s')
print('Validation metrics:', {key: round(value, 6) if isinstance(value, float) else value for key, value in selected_validation_metrics.items()})
print('\nMAE by pickup hour (minutes):')
print(pd.Series(hour_mae, index=range(24), name='MAE').to_string())
print('\nMAE by distance bucket (minutes):')
print(pd.Series(distance_mae, index=distance_labels, name='MAE').to_string())
print(f'Strongest hour: {strongest_hour}:00 (MAE {hour_mae[strongest_hour]:.6f} minutes)')
print(f'Weakest hour: {weakest_hour}:00 (MAE {hour_mae[weakest_hour]:.6f} minutes)')
print(f'Strongest distance bucket: {strongest_distance_bucket} (MAE {np.nanmin(distance_mae):.6f} minutes)')
print(f'Weakest distance bucket: {weakest_distance_bucket} (MAE {np.nanmax(distance_mae):.6f} minutes)')
print('\nValidation duration percentiles (minutes):')
for percentile, value in duration_percentiles.items():
    print(f'p{percentile}: {value:.6f}')
print(f'Maximum validation duration: {maximum_validation_duration:.6f} minutes')
print(f'Trips over {very_long_threshold_minutes:.0f} minutes: {very_long_count:,} ({very_long_percentage:.6f}%)')
print('Long-duration observations retained unchanged: YES')
print('Interpretation — predicted vs actual: predictions track typical trips, while the longest durations show substantial unexplained variation.')
print('Interpretation — residuals: the central residual mass is near zero, but a long positive tail contributes strongly to RMSE.')
print(f'Interpretation — hour: lowest MAE occurs at {strongest_hour}:00 and highest at {weakest_hour}:00.')
print(f'Interpretation — distance: lowest MAE occurs for {strongest_distance_bucket} and highest for {weakest_distance_bucket}.')
print('Limitations: long-duration outliers strongly affect RMSE; traffic, congestion, weather, and road conditions are absent; distance is conditionally available.')
print('Test data accessed in Step 10: NO')
print('Step 10 status: PASS')

| Model | MAE (minutes) | RMSE (minutes) | R² | Training Time |
|---|---:|---:|---:|---:|
| Median baseline | 9.756819 | 26.446473 | -0.024739 | n/a |
| LinearRegression without distance | 8.566630 | 24.971912 | 0.086347 | 3.630s |
| RandomForestRegressor | 8.708620 | 25.046847 | 0.080855 | 152.079s |
| ExtraTreesRegressor | 8.696595 | 25.090268 | 0.077666 | 144.491s |
| LinearRegression + conditional distance_miles | 6.367238 | 23.700579 | 0.177008 | 4.286s |

Selected model: LinearRegression with conditional distance_miles
Strict pre-trip fallback: LinearRegression without distance_miles
Selection condition: an estimated route distance is available before departure.
Validation model refit time for error analysis: 4.040s
Validation metrics: {'mae': 6.367238, 'rmse': 23.700579, 'r2': 0.177008, 'rows': 6814901}

MAE by pickup hour (minutes):
0     5.330875
1     4.946503
2     4.798572
3     5.123812
4     7.120411
5     8.602170
6     8.450204
7     7.502794
8     6.754376
9     6.4593

## Step 11 — Final retraining and one-time test evaluation

Model selection is complete. The selected route-estimate LinearRegression pipeline is now fitted on deterministic samples from both train and validation. The test split is opened for one full streaming performance pass only. No feature, parameter, cleaning, or split decision will change after observing the test metrics.

Chronological generalization is assessed using a stability rule declared before test access: absolute relative MAE and RMSE changes within 10%, and absolute R² change within 0.05.

In [16]:
# This guard is stored in the locked False state after the single successful execution.
STEP11_ALLOW_DURATION_TEST = False
if not STEP11_ALLOW_DURATION_TEST:
    raise RuntimeError('Duration test evaluation already completed. Do not rerun or unlock.')

# Reuse the 445,000 train sample and add 2,500 evenly spaced rows per validation row group.
validation_parquet_sample = pq.ParquetFile(VALIDATION_PATH)
validation_sample_tables = []
validation_sample_rows = 0
for row_group_index in range(validation_parquet_sample.num_row_groups):
    table = validation_parquet_sample.read_row_group(row_group_index, columns=FINAL_FEATURES + [TARGET])
    sample_size = min(SAMPLE_PER_ROW_GROUP, table.num_rows)
    indices = np.linspace(0, table.num_rows - 1, num=sample_size, dtype=np.int64)
    validation_sample_tables.append(table.take(pa.array(indices)))
    validation_sample_rows += sample_size
validation_parquet_sample.close()
validation_training_sample = pa.concat_tables(validation_sample_tables).to_pandas()
del validation_sample_tables

sampled_train_rows = len(step10_training_sample)
sampled_validation_rows = len(validation_training_sample)
assert sampled_train_rows == 445_000
assert sampled_validation_rows == 97_500
final_training_sample = pd.concat([step10_training_sample, validation_training_sample], ignore_index=True)
del validation_training_sample
assert len(final_training_sample) == 542_500

final_duration_pipeline = Pipeline([
    ('preprocessor', make_preprocessor()),
    ('model', LinearRegression()),
])
final_fit_start = time.perf_counter()
final_duration_pipeline.fit(
    final_training_sample[FINAL_FEATURES],
    final_training_sample[TARGET].to_numpy(dtype='float64'),
)
final_training_time = time.perf_counter() - final_fit_start

print('MODEL SELECTION COMPLETE: confirmed')
print('NO FURTHER TUNING: confirmed')
print('TEST WILL BE USED ONCE ONLY: confirmed')
print('Sampled train rows:', sampled_train_rows)
print('Sampled validation rows:', sampled_validation_rows)
print('Total final-training rows:', len(final_training_sample))
print(f'Final training time: {final_training_time:.3f} seconds')

# Single authorized open and a single full streaming evaluation pass.
duration_test_open_count = 0
duration_test_stream_passes = 0
duration_test_open_count += 1
test_parquet_11 = pq.ParquetFile(TEST_PATH)
assert test_parquet_11.metadata.num_rows == EXPECTED_TEST_ROWS
assert set(FINAL_FEATURES + [TARGET]).issubset(test_parquet_11.schema_arrow.names)
duration_test_stream_passes += 1
test_rows_11 = 0
test_absolute_error_sum = 0.0
test_squared_error_sum = 0.0
test_target_sum = 0.0
test_target_squared_sum = 0.0
for row_group_index in range(test_parquet_11.num_row_groups):
    frame = test_parquet_11.read_row_group(row_group_index, columns=FINAL_FEATURES + [TARGET]).to_pandas()
    actual = frame[TARGET].to_numpy(dtype='float64')
    prediction = final_duration_pipeline.predict(frame[FINAL_FEATURES])
    errors = prediction - actual
    test_rows_11 += len(actual)
    test_absolute_error_sum += float(np.abs(errors).sum())
    test_squared_error_sum += float(np.square(errors).sum())
    test_target_sum += float(actual.sum())
    test_target_squared_sum += float(np.square(actual).sum())
test_parquet_11.close()

assert duration_test_open_count == 1
assert duration_test_stream_passes == 1
assert test_rows_11 == EXPECTED_TEST_ROWS
test_sst_11 = test_target_squared_sum - test_target_sum**2 / test_rows_11
duration_test_metrics = {
    'mae': test_absolute_error_sum / test_rows_11,
    'rmse': float(np.sqrt(test_squared_error_sum / test_rows_11)),
    'r2': 1.0 - test_squared_error_sum / test_sst_11,
    'rows': test_rows_11,
}
test_minus_validation = {
    metric: duration_test_metrics[metric] - selected_validation_metrics[metric]
    for metric in ['mae', 'rmse', 'r2']
}
test_percent_change = {
    'mae': test_minus_validation['mae'] / selected_validation_metrics['mae'] * 100.0,
    'rmse': test_minus_validation['rmse'] / selected_validation_metrics['rmse'] * 100.0,
}
generalization_stable = (
    abs(test_percent_change['mae']) <= 10.0
    and abs(test_percent_change['rmse']) <= 10.0
    and abs(test_minus_validation['r2']) <= 0.05
)

print('| Dataset | MAE (minutes) | RMSE (minutes) | R² | Rows |')
print('|---|---:|---:|---:|---:|')
print(f"| Validation | {selected_validation_metrics['mae']:.6f} | {selected_validation_metrics['rmse']:.6f} | {selected_validation_metrics['r2']:.6f} | {selected_validation_metrics['rows']:,} |")
print(f"| Final Test | {duration_test_metrics['mae']:.6f} | {duration_test_metrics['rmse']:.6f} | {duration_test_metrics['r2']:.6f} | {duration_test_metrics['rows']:,} |")
print('Test minus validation MAE:', f"{test_minus_validation['mae']:+.6f} minutes ({test_percent_change['mae']:+.3f}%)")
print('Test minus validation RMSE:', f"{test_minus_validation['rmse']:+.6f} minutes ({test_percent_change['rmse']:+.3f}%)")
print('Test minus validation R²:', f"{test_minus_validation['r2']:+.6f}")
print('Chronological generalization stable under predeclared rule:', generalization_stable)
print('No model changes will be made after observing test metrics: confirmed')
print('TEST USED ONCE ONLY: confirmed')
print('Step 11 status: PASS')

MODEL SELECTION COMPLETE: confirmed
NO FURTHER TUNING: confirmed
TEST WILL BE USED ONCE ONLY: confirmed
Sampled train rows: 445000
Sampled validation rows: 97500
Total final-training rows: 542500
Final training time: 5.472 seconds
| Dataset | MAE (minutes) | RMSE (minutes) | R² | Rows |
|---|---:|---:|---:|---:|
| Validation | 6.367238 | 23.700579 | 0.177008 | 6,814,901 |
| Final Test | 5.802654 | 22.259013 | 0.206474 | 6,730,257 |
Test minus validation MAE: -0.564584 minutes (-8.867%)
Test minus validation RMSE: -1.441566 minutes (-6.082%)
Test minus validation R²: +0.029466
Chronological generalization stable under predeclared rule: True
No model changes will be made after observing test metrics: confirmed
TEST USED ONCE ONLY: confirmed
Step 11 status: PASS


### Step 11 interpretation

Chronological test performance improved relative to validation: MAE decreased by 0.564584 minutes (8.867%), RMSE decreased by 1.441566 minutes (6.082%), and R² increased by 0.029466. This meets the stability rule declared before test access. No feature, parameter, model, cleaning, or split change will be made after observing these one-time test metrics.


## Step 12 — Serialization and handover

The fitted Step 11 pipeline is serialized directly, reloaded, and checked against non-test training rows. This step does not reopen the test split or retrain the estimator.

In [17]:
MODEL_PATH = MODEL_DIR / 'duration_pipeline.pkl'
HANDOVER_PATH = DOCS_DIR / 'MEMBER2_DURATION_HANDOVER.md'

sanity_features = final_training_sample[FINAL_FEATURES].iloc[:8].copy()
predictions_before_serialization = final_duration_pipeline.predict(sanity_features)
joblib.dump(final_duration_pipeline, MODEL_PATH)
loaded_duration_pipeline = joblib.load(MODEL_PATH)
predictions_after_serialization = loaded_duration_pipeline.predict(sanity_features)

reload_preprocessor_ok = isinstance(
    loaded_duration_pipeline.named_steps.get('preprocessor'), ColumnTransformer
)
reload_model_ok = isinstance(
    loaded_duration_pipeline.named_steps.get('model'), LinearRegression
)
sanity_predictions_ok = (
    predictions_after_serialization.shape == (len(sanity_features),)
    and np.issubdtype(predictions_after_serialization.dtype, np.number)
    and np.isfinite(predictions_after_serialization).all()
    and np.allclose(predictions_before_serialization, predictions_after_serialization, rtol=0, atol=1e-12)
)
model_size_bytes = MODEL_PATH.stat().st_size
assert reload_preprocessor_ok and reload_model_ok and sanity_predictions_ok and model_size_bytes > 0

hour_rows = '\n'.join(f'| {hour:02d}:00 | {hour_mae[hour]:.6f} |' for hour in range(24))
distance_rows = '\n'.join(
    f'| {label} | {mae:.6f} |' for label, mae in zip(distance_labels, distance_mae)
)
handover_text = f"""# Member 2 Duration Prediction Handover

## A. Objective

Predict `trip_duration_minutes` before trip start.

## B. Final model

`LinearRegression` inside an sklearn `Pipeline`, using the route-estimate configuration.

## C. Final features

Numeric:

- `pickup_hour`
- `month`
- `distance_miles`

Categorical:

- `provider_code`
- `day_of_week`
- `weekend`
- `origin_loc_id`
- `dest_loc_id`

## D. Preprocessing

- Median numeric imputation
- Most-frequent categorical imputation
- `OneHotEncoder(handle_unknown="ignore")`
- `ColumnTransformer`

## E. Validation results

- MAE: {selected_validation_metrics['mae']:.6f} minutes
- RMSE: {selected_validation_metrics['rmse']:.6f} minutes
- R²: {selected_validation_metrics['r2']:.6f}
- Rows: {selected_validation_metrics['rows']:,}

## F. Test results

- MAE: {duration_test_metrics['mae']:.6f} minutes
- RMSE: {duration_test_metrics['rmse']:.6f} minutes
- R²: {duration_test_metrics['r2']:.6f}
- Rows: {duration_test_metrics['rows']:,}

The duration test split was evaluated once after model selection. No modelling decision changed afterward.

## G. Training data used

- Sampled train rows: {sampled_train_rows:,}
- Sampled validation rows: {sampled_validation_rows:,}
- Total final-training rows: {len(final_training_sample):,}
- Final training time: {final_training_time:.3f} seconds

## H. Model comparisons

| Model | Validation MAE | Validation RMSE | Validation R² | Training Time |
|---|---:|---:|---:|---:|
| Median baseline | 9.756819 | 26.446473 | -0.024739 | n/a |
| LinearRegression without distance | 8.566630 | 24.971912 | 0.086347 | 3.630s |
| RandomForestRegressor | 8.708620 | 25.046847 | 0.080855 | 152.079s |
| ExtraTreesRegressor | 8.696595 | 25.090268 | 0.077666 | 144.491s |
| LinearRegression with conditional distance | {selected_validation_metrics['mae']:.6f} | {selected_validation_metrics['rmse']:.6f} | {selected_validation_metrics['r2']:.6f} | 4.286s |

## I. Feature experiments

- `route_id` was excluded because one-hot preprocessing produced 22,392 encoded features, exceeding the 5,000-feature safety threshold.
- Conditional `distance_miles` improved validation MAE by 2.199392 minutes (25.674%) and RMSE by 1.271333 minutes (5.091%).

## J. Error analysis

### MAE by pickup hour

| Pickup hour | MAE (minutes) |
|---|---:|
{hour_rows}

Lowest hourly MAE occurred at {strongest_hour}:00 ({hour_mae[strongest_hour]:.6f} minutes); highest occurred at {weakest_hour}:00 ({hour_mae[weakest_hour]:.6f} minutes).

### MAE by estimated distance

| Distance bucket | MAE (minutes) |
|---|---:|
{distance_rows}

The strongest distance bucket was {strongest_distance_bucket}; the weakest was {weakest_distance_bucket}.

### Long-duration observations

- p50: {duration_percentiles[50]:.6f} minutes
- p90: {duration_percentiles[90]:.6f} minutes
- p95: {duration_percentiles[95]:.6f} minutes
- p99: {duration_percentiles[99]:.6f} minutes
- p99.5: {duration_percentiles[99.5]:.6f} minutes
- p99.9: {duration_percentiles[99.9]:.6f} minutes
- Maximum: {maximum_validation_duration:.6f} minutes
- Trips over {very_long_threshold_minutes:.0f} minutes: {very_long_count:,} ({very_long_percentage:.6f}%)

These observations were retained unchanged. Their large residuals strongly influence RMSE.

## K. Important limitations

- `distance_miles` is valid only when it represents an estimated route distance available before departure. Completed-trip mileage must not be treated as a pre-trip input.
- Long-duration outliers strongly affect RMSE.
- Traffic, congestion, weather, incidents, and road conditions are not represented by the current pre-trip features.

## L. Output artifacts

- `models/duration_pipeline.pkl`
- `notebooks/03_DurationPrediction.ipynb`
- `reports/figures/duration_predicted_vs_actual.png`
- `reports/figures/duration_residual_distribution.png`
- `reports/figures/duration_mae_by_hour.png`
- `reports/figures/duration_mae_by_distance_bucket.png`
"""
HANDOVER_PATH.write_text(handover_text, encoding='utf-8')

duration_figure_paths = [
    FIGURE_DIR / 'duration_predicted_vs_actual.png',
    FIGURE_DIR / 'duration_residual_distribution.png',
    FIGURE_DIR / 'duration_mae_by_hour.png',
    FIGURE_DIR / 'duration_mae_by_distance_bucket.png',
]
figures_verified = all(path.is_file() and path.stat().st_size > 0 for path in duration_figure_paths)
assert figures_verified and HANDOVER_PATH.is_file() and HANDOVER_PATH.stat().st_size > 0

print('duration_pipeline.pkl created: PASS')
print('Reload preprocessor is ColumnTransformer:', reload_preprocessor_ok)
print('Reload model is LinearRegression:', reload_model_ok)
print('Sanity predictions finite and matching:', sanity_predictions_ok)
print('Sanity prediction count:', len(predictions_after_serialization))
print('Sanity prediction values:', np.round(predictions_after_serialization, 6).tolist())
print('Model size (bytes):', model_size_bytes)
print('Handover created: PASS')
print('Four figures verified: PASS')
print('Test reopened during Step 12: NO')
print('Step 12 status: PASS')

duration_pipeline.pkl created: PASS
Reload preprocessor is ColumnTransformer: True
Reload model is LinearRegression: True
Sanity predictions finite and matching: True
Sanity prediction count: 8
Sanity prediction values: [35.377624, 43.731285, 7.650899, 10.249623, 26.327567, 10.304498, 20.988106, 31.458084]
Model size (bytes): 9580
Handover created: PASS
Four figures verified: PASS
Test reopened during Step 12: NO
Step 12 status: PASS


## Final duration modelling summary

The selected route-estimate model is LinearRegression with eight inputs and preprocessing contained inside the sklearn pipeline. It is deployable only when `distance_miles` is an estimated route distance available before trip start; the strict fallback excludes distance. Long-duration trips were retained unchanged, and their residuals materially increase RMSE. The one-time chronological test evaluation is complete and locked.

In [18]:
print('STEP 10 — PASS')
print('Selected model: LinearRegression with conditional distance_miles')
print('Selected features:', FINAL_FEATURES)
print(f"Validation MAE={selected_validation_metrics['mae']:.6f}, RMSE={selected_validation_metrics['rmse']:.6f}, R²={selected_validation_metrics['r2']:.6f}")
print(f'Strongest error finding: lowest MAE at {strongest_hour}:00 and in {strongest_distance_bucket}.')
print(f'Long-duration finding: maximum={maximum_validation_duration:.6f} minutes; over 120 minutes={very_long_count:,} ({very_long_percentage:.6f}%).')
print('Main limitation: long-duration tails and missing traffic/context variables; distance availability is conditional.')

print('\nSTEP 11 — PASS')
print(f'Train sample rows={sampled_train_rows:,}; validation sample rows={sampled_validation_rows:,}; total={len(final_training_sample):,}')
print(f'Final training time={final_training_time:.3f}s')
print(f"Test rows={duration_test_metrics['rows']:,}; MAE={duration_test_metrics['mae']:.6f}; RMSE={duration_test_metrics['rmse']:.6f}; R²={duration_test_metrics['r2']:.6f}")
print(f"Test minus validation: MAE={test_minus_validation['mae']:+.6f}, RMSE={test_minus_validation['rmse']:+.6f}, R²={test_minus_validation['r2']:+.6f}")
print('Test used once only: YES')

print('\nSTEP 12 — PASS')
print('duration_pipeline.pkl created: PASS')
print('Reload verification: PASS')
print('Sanity predictions: PASS')
print(f'Model size: {model_size_bytes:,} bytes')
print('Handover created: PASS')
print('Four figures verified: PASS')
print('Overall Steps 10–12: PASS')
print('STOP: no commit or push performed.')

STEP 10 — PASS
Selected model: LinearRegression with conditional distance_miles
Selected features: ['pickup_hour', 'month', 'distance_miles', 'provider_code', 'day_of_week', 'weekend', 'origin_loc_id', 'dest_loc_id']
Validation MAE=6.367238, RMSE=23.700579, R²=0.177008
Strongest error finding: lowest MAE at 2:00 and in 0–2 miles.
Long-duration finding: maximum=8611.866667 minutes; over 120 minutes=9,158 (0.134382%).
Main limitation: long-duration tails and missing traffic/context variables; distance availability is conditional.

STEP 11 — PASS
Train sample rows=445,000; validation sample rows=97,500; total=542,500
Final training time=5.472s
Test rows=6,730,257; MAE=5.802654; RMSE=22.259013; R²=0.206474
Test minus validation: MAE=-0.564584, RMSE=-1.441566, R²=+0.029466
Test used once only: YES

STEP 12 — PASS
duration_pipeline.pkl created: PASS
Reload verification: PASS
Sanity predictions: PASS
Model size: 9,580 bytes
Handover created: PASS
Four figures verified: PASS
Overall Steps 10–1